## Imports

In [1]:
"""
Initial imports and function declarations
"""

import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tqdm.notebook import tqdm, trange
import firedrake as fd
from firedrake import (
    eq,
    conditional,
    min_value,
    max_value,
    exp,
    sqrt,
    inner,
    sym,
    tr,
    grad,
    Constant,
    dx,
    ds,
    dS,
    avg,
    jump,
)
import irksome
from irksome import Dt, lag
from icepack.constants import (
    weertman_sliding_law,
    glen_flow_law,
    gravity as g,
    ice_density as ρ_I,
    water_density as ρ_W,
)

import ufl

firedrake:WARNING OMP_NUM_THREADS is not set or is set to a value greater than 1, we suggest setting OMP_NUM_THREADS=1 to improve performance


## Definition of Laws

In [2]:
# def get_test_function(q):
#     z, = ufl.algorithms.extract_coefficients(q)
#     w = fd.TestFunction(z.function_space())
#     return fd.replace(q, {z: w})


# # redundant, use the imports instead
# n = 3 #Constant(glen_flow_law)
# m = 3 #Constant(weertman_sliding_law)

# # fields are mutable state?
# # parameters are immutable?
# # This uses the global variable s, which is mutable. Should it be part of fields?
# def friction_law(**kwargs):
#     τ, u, h = map(kwargs.get, ("basal_stress", "velocity", "thickness"))

#     parameters = ("sliding_coefficient", "sliding_exponent")
#     K, m = map(kwargs.get, parameters)

#     p_W = lag(ρ_W * g * max_value(0, -(s - h)))
#     p_I = ρ_I * g * h
#     r = Constant(0.99)
#     ϕ = lag(conditional(p_W > r * p_I, 0, 1))

#     σ = get_test_function(τ)
#     τ_2 = inner(τ, τ)

#     if m == 1:
#         τ_m = Constant(1.0)
#     elif m == 3:
#         τ_m = τ_2
#     elif m == 5:
#         τ_m = τ_2 ** 2
#     else:
#         raise ValueError("Sliding law exponent must be in [1, 3, 5]!")

#     f = ϕ if m == 1 else ϕ ** m
#     return inner(K * τ_m * τ + f * u, σ) * dx


# def flow_law(**kwargs):
#     variables = ("velocity", "membrane_stress", "thickness")
#     u, M, h = map(kwargs.get, variables)
#     parameters = ("flow_law_coefficient", "flow_law_exponent")
#     A, n = map(kwargs.get, parameters)

#     N = get_test_function(M)
#     d = mesh.geometric_dimension
#     M_2 = (inner(M, M) - tr(M)**2 / (d + 1)) / 2

#     if n == 1:
#         M_n = Constant(1.0)
#     elif n == 3:
#         M_n = M_2
#     elif n == 5:
#         M_n = M_2 ** 2
#     else:
#         raise ValueError("Flow law exponent must be in [1, 3, 5]!")

#     ε = sym(grad(u))
#     return h * (A * M_n * (inner(M, N) - tr(M) * tr(N) / (d + 1)) - inner(ε, N)) * dx


# def momentum_balance(**kwargs):
#     variables = (
#         "velocity", "membrane_stress", "basal_stress", "thickness", "surface"
#     )
#     u, M, τ, h, s = map(kwargs.get, variables)
#     v = get_test_function(u)

#     ε = sym(grad(v))
#     cell_balance = (-h * inner(M, ε) + inner(τ - ρ_I * g * h * grad(s), v)) * dx

#     ν = fd.FacetNormal(mesh)
#     facet_balance = ρ_I * g * avg(h) * inner(jump(s, ν), avg(v)) * dS

#     return cell_balance + facet_balance


# def terminus(**kwargs):
#     variables = ("velocity", "thickness", "surface", "terminus_ids")
#     u, h, s, terminus_ids = map(kwargs.get, variables)
#     v = get_test_function(u)

#     d = fd.min_value(s - h, 0)
#     τ_I = ρ_I * g * h**2 / 2
#     τ_W = ρ_W * g * d**2 / 2

#     ν = fd.FacetNormal(mesh)
#     return (τ_I - τ_W) * inner(v, ν) * ds(terminus_ids)


# def smooth_max(a, b, ϵ):
#     return (a + b + ((a - b)**2 + ϵ**2) ** 0.5) / 2

# def mass_balance(**kwargs):
#     variables = ("thickness", "velocity", "accumulation", "thickness_in")
#     h, u, a, h_in = map(kwargs.get, variables)
#     η = get_test_function(h)

#     ν = fd.FacetNormal(mesh)
#     F_cells = (Dt(h) * η - inner(h * u, grad(η)) - a * η) * dx
#     f = h * max_value(0, inner(u, ν))
#     F_facets = jump(f) * jump(η) * dS
#     F_outflow = f * η * ds
#     F_inflow = h_in * min_value(0, inner(u, ν)) * η * ds
#     return F_cells + F_facets + F_outflow + F_inflow

## Initialization and setting the values

In [3]:
# """
# Initialization of the bed geometry and velocity/height profiles
# """

# lx = 610e3
# nx = 450

# thickness_element = fd.FiniteElement("DG", "interval", 0)
# bed_element = fd.FiniteElement("CG", "interval", 1)
# degree = 1

# velocity_element = fd.FiniteElement("CG", "interval", degree)
# stress_element = fd.FiniteElement("DG", "interval", degree - 1)

# Q = fd.FunctionSpace(mesh, thickness_element)
# S = fd.FunctionSpace(mesh, bed_element)
# V = fd.VectorFunctionSpace(mesh, velocity_element)
# Σ = fd.TensorFunctionSpace(mesh, stress_element, symmetry=True)

# Lx = Constant(lx)

In [4]:
# x = fd.SpatialCoordinate(mesh)[0]

# b = fd.Function(S).interpolate(mismip_bed(mesh))

# h_in = Constant(200.0)
# δh = Constant(100.0)
# h_expr = h_in - δh * x / Lx
# h_0 = fd.Function(Q).interpolate(h_expr)
# h = h_0.copy(deepcopy=True)

# ϵ = Constant(0.1)
# s_expr = smooth_max(b + h, (1 - ρ_I / ρ_W) * h, ϵ)
# s0 = fd.Function(Q).interpolate(s_expr)
# z_b = fd.Function(Q).interpolate(s0 - h)

In [5]:
# δu = fd.Constant(90.0)
# expr = fd.as_vector([δu * x / Lx])  # 1-component vector
# u_0 = fd.Function(V).interpolate(expr)

In [6]:
# A = Constant(20)
# K = Constant(1e6)

# τ_c = Constant(0.01)
# ε_c = Constant(A * τ_c ** n)
# u_c = Constant(K * τ_c ** m)

In [7]:
# print(f"Strain rate: {1000 * float(ε_c):0.3f} m / yr / km")
# print(f"Speed:       {float(u_c):0.3f} m / yr")
# print(f"at stress of {1000 * float(τ_c):0.3f} kPa")

In [8]:
# """
# Initialization of function spaces and mass/momentum balance
# """

# Z = V * Σ * V
# z = fd.Function(Z)
# z.sub(0).assign(u_0);

# inflow_ids = (1,)
# terminus_ids = (2,)

# #s = smooth_max(b + h, (1 - ρ_I / ρ_W) * h, ϵ)
# s = max_value(b + h, (1 - ρ_I / ρ_W) * h)

# u, M, τ = fd.split(z)

# # Mutable State (partially derived from z?)?
# fields = {
#     "velocity": u,
#     "thickness": h,
#     "surface": s,
#     "membrane_stress": M,
#     "basal_stress": τ,
# }

# parameters_1 = {
#     "flow_law_coefficient": ε_c / τ_c,
#     "flow_law_exponent": 1,
#     "sliding_coefficient": u_c / τ_c,
#     "sliding_exponent": 1,
# }

# parameters_3 = {
#     "flow_law_coefficient": ε_c / τ_c ** n,
#     "flow_law_exponent": n,
#     "sliding_coefficient": u_c / τ_c ** m,
#     "sliding_exponent": m,
# }

# α = Constant(1e-3)
# F_flow_law_1 = flow_law(**fields, **parameters_1)
# F_flow_law_3 = flow_law(**fields, **parameters_3)
# F_flow_law = (1 / (1 + α) * F_flow_law_3 + α / (1 + α) * F_flow_law_1)

# F_sliding_law_1 = friction_law(**fields, **parameters_1)
# F_sliding_law_3 = friction_law(**fields, **parameters_3)
# F_sliding_law = (1 / (1 + α) * F_sliding_law_3 + α / (1 + α) * F_sliding_law_1)

# F_balance = momentum_balance(**fields)
# F_terminus = terminus(**fields, terminus_ids=terminus_ids)

# F = F_flow_law + F_sliding_law + F_balance + F_terminus

# inflow_bc = fd.DirichletBC(Z.sub(0), u_0, inflow_ids)
# bcs = [inflow_bc]

## Loading the Steady State

In [9]:
import h5py

In [10]:
with h5py.File('mismip-steady-state.h5', 'r') as f:
    print("--- Available functions in this file ---")
    if 'functions' in f:
        f['functions'].visititems(lambda name, obj: print(f"functions/{name}") if isinstance(obj, h5py.Group) else None)
    else:
        print("No 'functions' group found. Let's look at the whole file map:")
        f.visititems(lambda name, obj: print(name) if isinstance(obj, h5py.Group) else None)

--- Available functions in this file ---
No 'functions' group found. Let's look at the whole file map:
topologies
topologies/DM_0xba3219500_0
topologies/DM_0xba3219500_0/distributions
topologies/DM_0xba3219500_0/distributions/firedrake_default_1_True_simple_(FACET,1)
topologies/DM_0xba3219500_0/distributions/firedrake_default_1_True_simple_(FACET,1)/permutations
topologies/DM_0xba3219500_0/distributions/firedrake_default_1_True_simple_(FACET,1)/permutations/firedrake_default_False
topologies/DM_0xba3219500_0/dms
topologies/DM_0xba3219500_0/dms/coordinateDM
topologies/DM_0xba3219500_0/dms/coordinateDM/section
topologies/DM_0xba3219500_0/dms/coordinateDM/section/field0
topologies/DM_0xba3219500_0/dms/coordinateDM/section/field0/component0
topologies/DM_0xba3219500_0/dms/coordinateDM/vecs
topologies/DM_0xba3219500_0/dms/coordinateDM/vecs/coordinates
topologies/DM_0xba3219500_0/dms/firedrake_dm_0_1_False_1
topologies/DM_0xba3219500_0/dms/firedrake_dm_0_1_False_1/section
topologies/DM_0xba3

In [11]:
# load steady state data
with fd.CheckpointFile('steady-state.h5', "r") as chk:
    mesh = chk.load_mesh("mesh")

In [12]:
lx = 610e3
nx = 450

thickness_element = fd.FiniteElement("DG", "interval", 0)
bed_element = fd.FiniteElement("CG", "interval", 1)
degree = 1

velocity_element = fd.FiniteElement("CG", "interval", degree)
stress_element = fd.FiniteElement("DG", "interval", degree - 1)

Q = fd.FunctionSpace(mesh, thickness_element)
S = fd.FunctionSpace(mesh, bed_element)
V = fd.VectorFunctionSpace(mesh, velocity_element)
Σ = fd.TensorFunctionSpace(mesh, stress_element, symmetry=True)

In [13]:
W = V * Σ * V * Q
w = fd.Function(W)
b = fd.Function(S)

In [14]:
with fd.CheckpointFile('mismip-steady-state.h5', "r") as chk:
    b.vector()[:] = chk._load_function_topology(
        S.mesh(), 
        S.ufl_element(), 
        "bed",
    )

Error: error code 79
[0] DMPlexSectionLoad() at /Users/ada/Code/firedrake/petsc/src/dm/impls/plex/plex.c:2657
[0] DMPlexSectionLoad_HDF5_Internal() at /Users/ada/Code/firedrake/petsc/src/dm/impls/plex/hdf5/plexhdf5.c:2981
[0] ISLoad() at /Users/ada/Code/firedrake/petsc/src/vec/is/is/interface/index.c:1667
[0] ISLoad_Default() at /Users/ada/Code/firedrake/petsc/src/vec/is/utils/isio.c:113
[0] ISLoad_HDF5() at /Users/ada/Code/firedrake/petsc/src/vec/is/utils/isio.c:50
[0] PetscViewerHDF5Load() at /Users/ada/Code/firedrake/petsc/src/vec/is/utils/hdf5/hdf5io.c:327
[0] PetscViewerHDF5Load_Internal() at /Users/ada/Code/firedrake/petsc/src/vec/is/utils/hdf5/hdf5io.c:246
[0] Unexpected data in file
[0] Object (dataset) "order" not stored in group /topologies/mesh/dms/firedrake_dm_1_0_False_1

In [ ]:
with fd.CheckpointFile('steady-state.h5', "r") as chk:
    b = chk.load_function(mesh, name="bed")
    w = chk.load_function(mesh, name="mono")

RuntimeError: 
                Function (bed) not found under either of the following path in steady-state.h5:

                topologies/firedrake_mixed_meshes/mesh
                topologies/DM_0x92b8b6140_0/firedrake_meshes/mesh
            

## Functions for Determining the Terminus

In [ ]:
def end_of_glacier_fast(f):
    """
    This function takes a (thickness) function f on a 1-D mesh and returns the first integer where f is zero.
    """
    f_values = f.dat.data_ro # if hasattr(f, "dat") else fd.Function(f).dat.data_ro
    # print(f_values)
    zeros = np.where(np.isclose(f_values, 0.0, atol=1e-12))[0]
    # print(zeros)

    if len(zeros) == 0: # if the this is the case then the terminus lies outside of our domain
        return 'empty'
    return int(zeros[0]*(610_000/450)) # this gives us the first point at which the thickness is zero

In [ ]:
def end_of_glacier_node(f):
    """
    This function takes a (thickness) function f on a 1-D mesh and returns the first node (of the mesh) where f is zero.
    """
    f_values = f.dat.data_ro # if hasattr(f, "dat") else fd.Function(f).dat.data_ro
    # print(f_values)
    zeros = np.where(np.isclose(f_values, 0.0, atol=1e-12))[0]
    # print(zeros)

    if len(zeros) == 0: # if the this is the case then the terminus lies outside of our domain
        return 'empty'
    return zeros[0]

## Calving Time Stepper

In [ ]:
"""
Main driver:
"""

method = irksome.BackwardEuler()
t = Constant(0.0)
timestep = 0.2
dt = Constant(timestep)

fparams = {"quadrature_degree": 8}
sparams = {
    "snes_monitor": ":mismip-dual-ts.log",
    "snes_type": "vinewtonrsls",
    "snes_linesearch_type": "bt", 
    "snes_linesearch_max_it": 200,
    "snes_max_it": 1000,
    "snes_atol": 2e-8,
    "snes_rtol": 1e-4,
    "pc_factor_mat_solver_type": "mumps",
}
lower = fd.Function(W)
upper = fd.Function(W)
lower.assign(-np.inf)
lower.sub(3).assign(0.0)
upper.assign(+np.inf)

params = {
    "bcs": bcs,
    "form_compiler_parameters": fparams,
    "solver_parameters": sparams,
    "stage_type": "value",
    "basis_type": "Bernstein",
    "bounds": ("stage", lower, upper)
}

In [ ]:
w1 = w

solver = irksome.TimeStepper(F, method, t, dt, w1, **params, Jp=Fp)

final_time = 3000
timestep = 0.2
num_steps = int(final_time / timestep)

ws = [w1.copy(deepcopy=True)]

for step in trange(num_steps):
    h = w1.sub(3)
    x = fd.SpatialCoordinate(fd.Function(h).ufl_domain())[0]

    f = fd.Function(Q)

    h_values = h.dat.data_ro
    small_thickness_node = np.where(h_values<=100)[0][0]
    print(small_thickness_node*(610_000/450))
    expr = h * fd.conditional(fd.le(x, small_thickness_node * (610_000/450)), 1.0, 0.0)
    f.interpolate(expr)
    w1.sub(3).assign(f)
     
    solver.advance()
    ws.append(w1.copy(deepcopy=True))